# 01. Data Check
**목적**: 날씨마루 Hive 및 제공 데이터의 구조를 확인하고, 파이프라인 분기 결정에 필요한 정보를 수집한다.

## 데이터 접근 경로 (확정)

| 경로 | 내용 |
|------|------|
| **날씨마루 Hive** | 기상 데이터, 전력설비 데이터, 화재 데이터 (SQL 쿼리) |
| **기상자료 개방포털** | 추가 기상 관측·기후통계 (필요 시) |
| **국가공간정보포털** | 산림/지형 공간 데이터 (공식 허용) |
| **산림빅데이터거래소** | 산불 발생 이력 (공식 허용) |

## 핵심 확인 항목
- Hive 테이블 목록 및 실제 컬럼명
- 화재 데이터(label) 존재 여부 → FAQ Q.49에서 제공 확인됨
- 기상 데이터 단위 → 일(daily) 자료 제공 확인됨
- 전력설비 데이터 형태 (point / line)

In [ ]:
import sys
sys.path.append('..')

import json
import pandas as pd
import numpy as np
from pathlib import Path
from config import (
    DATA_RAW, DATA_PROCESSED,
    HIVE_HOST, HIVE_PORT, HIVE_DATABASE, HIVE_USERNAME, HIVE_PASSWORD,
    HIVE_TABLES, ALLOW_EXTERNAL_SPATIAL
)

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

## 1. 날씨마루 Hive 연결

날씨마루는 Hive 기반 클라우드 분석 환경입니다.  
아래 두 가지 방법 중 환경에 맞는 것을 사용하세요.

In [ ]:
# ── 방법 A: pyhive 사용 (pip install pyhive thrift) ──
def connect_hive_pyhive():
    from pyhive import hive
    conn = hive.Connection(
        host=HIVE_HOST,
        port=HIVE_PORT,
        database=HIVE_DATABASE,
        username=HIVE_USERNAME,
        password=HIVE_PASSWORD,
        auth='CUSTOM'
    )
    return conn

# ── 방법 B: JayDeBeApi / JDBC 사용 (날씨마루 가이드 권장 방식) ──
def connect_hive_jdbc():
    import jaydebeapi
    # 날씨마루 하이브 매뉴얼의 JDBC URL 형식
    jdbc_url = f"jdbc:hive2://{HIVE_HOST}:{HIVE_PORT}/{HIVE_DATABASE}"
    conn = jaydebeapi.connect(
        "org.apache.hive.jdbc.HiveDriver",
        jdbc_url,
        [HIVE_USERNAME, HIVE_PASSWORD],
        "/path/to/hive-jdbc.jar"  # 날씨마루 제공 jar 경로로 교체
    )
    return conn

# ── 방법 C: 날씨마루 JupyterHub 환경 내부 (이미 연결된 경우) ──
# 날씨마루 JupyterHub에서 직접 실행하는 경우 SparkSession 또는
# 기존 conn 객체를 그대로 사용

print("연결 방법을 확인 후 아래 conn 변수에 할당하세요.")
print(f"HIVE_HOST: {HIVE_HOST}")
print(f"외부 공간데이터 사용: {ALLOW_EXTERNAL_SPATIAL}")

In [ ]:
# ── 날씨마루 접속 후 실제 연결 객체 할당 ──
# conn = connect_hive_pyhive()   # 또는 connect_hive_jdbc()

def hive_query(sql, conn=None):
    """Hive 쿼리 실행 → DataFrame 반환. conn이 None이면 로컬 CSV fallback."""
    if conn is not None:
        return pd.read_sql(sql, conn)
    else:
        # 로컬 테스트용: data/raw/ 에 CSV가 있으면 읽기
        print(f"[OFFLINE] conn 없음 — data/raw/ CSV fallback")
        return None

conn = None  # 날씨마루 접속 후 실제 conn으로 교체

## 2. 사용 가능한 Hive 테이블 목록 확인

In [ ]:
if conn is not None:
    df_tables = hive_query("SHOW TABLES", conn)
    print(f"=== 전체 테이블 수: {len(df_tables)} ===")
    print(df_tables.to_string())
else:
    print("[OFFLINE] 날씨마루 접속 후 실행하세요.")
    print("예상 테이블명 (FAQ Q.01 기준):")
    print("  기상 일자료    : db_sfc_obs_day")
    print("  기상 시간자료  : db_sfc_obs_tim")
    print("  전력설비       : (접속 후 확인 필요)")
    print("  화재 데이터    : (접속 후 확인 필요 — FAQ Q.49 시간자료 제공 확인)")

## 3. 기상 데이터 확인

In [ ]:
# ── 테이블명을 실제 이름으로 교체 ──
WEATHER_TABLE = HIVE_TABLES['weather']

if conn is not None:
    # 구조 확인
    df_w_schema = hive_query(f"DESCRIBE {WEATHER_TABLE}", conn)
    print(f"=== {WEATHER_TABLE} 컬럼 구조 ===")
    print(df_w_schema.to_string())

    # 샘플 10행
    df_w_sample = hive_query(f"SELECT * FROM {WEATHER_TABLE} LIMIT 10", conn)
    print("\n=== 샘플 데이터 ===")
    print(df_w_sample)

    # 기간 확인
    df_w_range = hive_query(
        f"SELECT MIN(tm) as min_date, MAX(tm) as max_date, COUNT(*) as row_cnt FROM {WEATHER_TABLE}",
        conn
    )
    print("\n=== 기간 및 행 수 ===")
    print(df_w_range)
else:
    # 로컬 CSV fallback
    csv_path = DATA_RAW / 'weather.csv'
    if csv_path.exists():
        df_w_sample = pd.read_csv(csv_path, encoding='utf-8-sig', nrows=5)
        print(df_w_sample)
    else:
        print("[OFFLINE] 날씨마루 접속 또는 data/raw/weather.csv 필요")

In [ ]:
# 핵심 기상 변수 존재 여부 체크
# FAQ 기준으로 예상되는 컬럼명
EXPECTED_WEATHER_COLS = {
    'ta':   '기온 (°C)',
    'hm':   '상대습도 (%)',
    'ws':   '풍속 (m/s)',
    'wd':   '풍향 (°)',
    'rn':   '강수량 (mm)',
    'efr':  '실효습도 (%)',      # 있으면 사용
    'stn':  '관측소 번호',
    'tm':   '일시',
    'lon':  '경도',
    'lat':  '위도',
}

print("=== 예상 기상 컬럼 목록 (날씨마루 표준) ===")
for col, desc in EXPECTED_WEATHER_COLS.items():
    print(f"  {col:8s} : {desc}")

print("\n※ 날씨마루 접속 후 DESCRIBE로 실제 컬럼명 확인 필요")
print("※ FAQ Q.17: 풍향은 16방위 → 0~360° 환산 가능")
print("※ FAQ Q.49: 기상 데이터는 일(daily) 자료로 제공")

## 4. 전력설비 데이터 확인

In [ ]:
FACILITY_TABLE = HIVE_TABLES['facility']

if conn is not None:
    df_f_schema = hive_query(f"DESCRIBE {FACILITY_TABLE}", conn)
    print(f"=== {FACILITY_TABLE} 컬럼 구조 ===")
    print(df_f_schema.to_string())

    df_f_sample = hive_query(f"SELECT * FROM {FACILITY_TABLE} LIMIT 10", conn)
    print("\n=== 샘플 데이터 ===")
    print(df_f_sample)

    df_f_count = hive_query(f"SELECT COUNT(*) as cnt FROM {FACILITY_TABLE}", conn)
    print(f"\n전체 설비 수: {df_f_count.iloc[0,0]:,}")
else:
    csv_path = DATA_RAW / 'facility.csv'
    if csv_path.exists():
        df_f_sample = pd.read_csv(csv_path, encoding='utf-8-sig', nrows=5)
        print(df_f_sample)
    else:
        print("[OFFLINE] 날씨마루 접속 또는 data/raw/facility.csv 필요")

In [ ]:
# 확인 항목 체크리스트
print("=== 전력설비 데이터 확인 항목 ===")
print("""
[ ] 좌표 컬럼 존재 여부 (위도/경도 또는 x/y)
[ ] 설비 형태 — point (변전소/변압기) or line (송전선/배전선)
[ ] 설비 유형 컬럼 (송전, 배전, 변전소 등)
[ ] 전압 등급 컬럼
[ ] 설치연도 컬럼 (노후도 feature 생성용)
[ ] 전력망 연결 정보 (from/to ID) — network centrality 가능 여부
""")

## 5. 화재 데이터 확인 (Label)
> **FAQ Q.49 확인**: 화재 데이터는 시간(hourly) 자료로 제공됨

In [ ]:
FIRE_TABLE = HIVE_TABLES['fire']

if conn is not None:
    df_fire_schema = hive_query(f"DESCRIBE {FIRE_TABLE}", conn)
    print(f"=== {FIRE_TABLE} 컬럼 구조 ===")
    print(df_fire_schema.to_string())

    df_fire_sample = hive_query(f"SELECT * FROM {FIRE_TABLE} LIMIT 10", conn)
    print("\n=== 샘플 데이터 ===")
    print(df_fire_sample)

    df_fire_count = hive_query(f"SELECT COUNT(*) as cnt FROM {FIRE_TABLE}", conn)
    print(f"\n전체 화재 건수: {df_fire_count.iloc[0,0]:,}")
else:
    csv_path = DATA_RAW / 'fire.csv'
    if csv_path.exists():
        df_fire_sample = pd.read_csv(csv_path, encoding='utf-8-sig', nrows=5)
        print(df_fire_sample)
    else:
        print("[OFFLINE] 날씨마루 접속 후 화재 테이블명 확인 필요")
        print("※ FAQ Q.49: '화재 데이터는 시간자료로 제공' — 테이블 존재 확인됨")

In [ ]:
# 화재 데이터 확인 항목
print("=== 화재 데이터 확인 항목 ===")
print("""
[ ] 화재 발생 위치 (위도/경도)
[ ] 화재 발생 일시 (시간자료 — hourly)
[ ] 화재 유형 (건축물 화재 / 산불 / 전력설비 인근 등)
[ ] 전력설비 ID와 직접 연결 가능한지 여부

label 정의 전략:
  Option A: 설비 반경 N km 내 화재 발생 여부 (binary label)
  Option B: 설비 반경 N km 내 화재 발생 건수 (count label → regression)
  권장: Option A (binary) — Recall@Top-K 평가에 최적
""")

## 6. 외부 공공데이터 확인
> 대회 "기타 데이터" 페이지 공식 링크 포털 — 사용 가능 확정

In [ ]:
from config import EXTERNAL_SOURCES

print("=== 공식 허용 외부 데이터 포털 ===")
EXTERNAL_DATA_PLAN = [
    {
        "source": "국가공간정보포털",
        "url": EXTERNAL_SOURCES['spatial'],
        "data": "산림청 임상도 (GIS 레이어)",
        "feature": "forest_ratio_Xm, distance_to_forest",
        "save_as": "data/external/forest.gpkg",
        "priority": "★★★★★"
    },
    {
        "source": "국가공간정보포털",
        "url": EXTERNAL_SOURCES['spatial'],
        "data": "수치표고모델 DEM",
        "feature": "elevation_mean_Xm, slope_mean_Xm",
        "save_as": "data/external/dem.tif",
        "priority": "★★★☆☆"
    },
    {
        "source": "산림빅데이터거래소",
        "url": EXTERNAL_SOURCES['forest_trade'],
        "data": "산불 발생 통계",
        "feature": "fire_count_Xkm, same_season_fire",
        "save_as": "data/external/fire_history.csv",
        "priority": "★★★★☆"
    },
    {
        "source": "재난안전데이터포털",
        "url": EXTERNAL_SOURCES['disaster'],
        "data": "화재 발생 현황",
        "feature": "external validation용",
        "save_as": "data/external/disaster_fire.csv",
        "priority": "★★★☆☆"
    },
    {
        "source": "SGIS plus",
        "url": EXTERNAL_SOURCES['sgis'],
        "data": "행정구역 경계 GIS",
        "feature": "지도 시각화용 행정구역 레이어",
        "save_as": "data/external/admin_boundary.gpkg",
        "priority": "★★☆☆☆"
    },
]

for item in EXTERNAL_DATA_PLAN:
    print(f"\n[{item['priority']}] {item['source']}")
    print(f"  데이터: {item['data']}")
    print(f"  생성 feature: {item['feature']}")
    print(f"  저장 경로: {item['save_as']}")

In [ ]:
# 외부 데이터 다운로드 현황 체크
from config import DATA_EXTERNAL
DATA_EXTERNAL.mkdir(parents=True, exist_ok=True)

external_files = [
    'forest.gpkg',
    'dem.tif',
    'fire_history.csv',
    'disaster_fire.csv',
    'admin_boundary.gpkg',
]

print("=== 외부 데이터 다운로드 현황 ===")
for f in external_files:
    path = DATA_EXTERNAL / f
    status = f"✓ ({path.stat().st_size/1024:.0f} KB)" if path.exists() else "✗ 미다운로드"
    print(f"  {f:30s}: {status}")

## 7. 파이프라인 분기 결정 및 column_map 저장

In [ ]:
# ── 날씨마루 접속 후 실제 컬럼명으로 교체 ──
column_map = {
    "weather": {
        "table":       HIVE_TABLES['weather'],
        "date":        "tm",        # 실제 컬럼명으로 교체
        "station_id":  "stn",
        "lat":         "lat",
        "lon":         "lon",
        "temperature": "ta",
        "humidity":    "hm",
        "wind_speed":  "ws",
        "wind_dir":    "wd",
        "precip":      "rn",
        "eff_humidity":"efr",       # 없으면 None
    },
    "facility": {
        "table":        HIVE_TABLES['facility'],
        "facility_id":  None,       # 접속 후 확인
        "lat":          None,
        "lon":          None,
        "type":         None,
        "voltage":      None,
        "install_year": None,
        "label":        None,       # 화재 데이터와 join 후 생성
    },
    "fire": {
        "table":     HIVE_TABLES['fire'],
        "datetime":  None,          # 접속 후 확인 (시간자료)
        "lat":       None,
        "lon":       None,
        "fire_type": None,
    },
    "flags": {
        "has_label":             True,    # FAQ Q.49: 화재 데이터 제공 확인
        "facility_geometry":     "point", # 접속 후 확인 후 수정
        "has_network":           False,   # 접속 후 확인 후 수정
        "weather_resolution":    "daily", # FAQ Q.49: 기상은 일자료
        "fire_resolution":       "hourly",# FAQ Q.49: 화재는 시간자료
        "allow_external_spatial":True,    # 대회 기타데이터 페이지 공식 허용
        "data_source":           "hive",  # hive | csv
    }
}

with open(DATA_PROCESSED / 'column_map.json', 'w', encoding='utf-8') as f:
    json.dump(column_map, f, ensure_ascii=False, indent=2)

print("column_map.json 저장 완료")
print()
FLAGS = column_map['flags']
print("=== 파이프라인 분기 결정 결과 ===")
print(f"  모델 유형       : Case A — Supervised ML (화재 label 제공 확인)")
print(f"  기상 해상도     : {FLAGS['weather_resolution']}")
print(f"  화재 데이터     : {FLAGS['fire_resolution']} (label 생성에 활용)")
print(f"  외부 공간데이터 : 허용 (기타데이터 페이지 공식 링크)")
print(f"  데이터 소스     : {FLAGS['data_source']}")
print()
print("⚠ 날씨마루 접속 후 실제 테이블명·컬럼명을 column_map.json에 업데이트하세요.")
print("다음 단계: 02_spatial_preprocessing.ipynb")